## Model hypeparametrization
Using Optuna and RandomSearch to finetune model parameters to improve prediction performance

In [208]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

from warnings import filterwarnings
filterwarnings('ignore')
from sklearn import set_config
set_config(display="text")


In [175]:
DATA_DIR = "../data"
RESULTS_DIR = "../results"

os.makedirs(f"{RESULTS_DIR}/model/optimization/", exist_ok=True)

### Load data

In [176]:
### Load merged df
df = pd.read_csv(f"{DATA_DIR}/preprocessed_df.csv")
df.head()

,record_id,sexo,fecha_ingreso,fecha_alta,foco_controlable,IRAs_nosocomial,mortalidad,fecha_mortalidad,dias_hemocultivo_mortalidad,mortalidad_30_dias,...,cateter_venoso,sonda_urinaria,sonda_nasogastrica,derivacion_ventriculoper,valvula_prot_cardiaca,portador_otros_disposit,antimicrobiano_list,cmi_list,interpretacion_list,antimicrobiano_family_list
0,1,0,2021-08-24,2021-09-09,NaN,1,1,2021-09-09,9.0,1,...,0,0,1,0,0,0,"['Cefazolina', 'Trimetoprim', 'Ertapenem', 'Ce...","['>16 mg/L', '>4 mg/L', '>1 mg/L', '>8 mg/L', ...","['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'I', ...","['Cefalosporin', 'TMP_SMX', 'Carbapenem', 'Cef..."
1,2,1,2023-06-18,2023-07-13,NaN,1,0,NaN,NaN,0,...,0,0,1,0,0,0,"['Trimetroprim/sulfametoxazol', 'Cefuroxima', ...","['>4/76 mg/L', '<=8 mg/L', '0.5 mg/L', '<=0.5 ...","['R', 'I', 'I', 'S', 'S', 'S', 'S', 'S', 'S', ...","['TMP_SMX', 'Cefalosporin', 'Quinolone', 'Quin..."
2,3,0,2022-02-11,2022-04-06,1.0,1,0,NaN,NaN,0,...,0,0,1,0,0,0,"['Trimetroprim/sulfametoxazol', 'Ampicilina', ...","['>4/76 mg/L', '>8 mg/L', '>4 mg/L', '>1 mg/L'...","['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', ...","['TMP_SMX', 'Penicillin', 'Aminoglycoside', 'Q..."
3,4,0,2021-05-15,2021-07-28,NaN,1,0,NaN,NaN,0,...,0,0,1,0,0,0,"['Trimetoprim', 'Levofloxacino', 'Ertapenem', ...","['>4 mg/L', '>1 mg/L', '>1 mg/L', '>16 mg/L', ...","['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', ...","['TMP_SMX', 'Quinolone', 'Carbapenem', 'Cefalo..."
4,5,1,2021-07-04,2022-01-04,0.0,1,0,NaN,NaN,0,...,0,0,0,0,0,0,"['Aztreonam', 'Ciprofloxacino', 'Ceftazidima',...","['<=1 mg/L', '>1 mg/L', '<=1 mg/L', '>8 mg/L',...","['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', ...","['Monobactam', 'Quinolone', 'Cefalosporin', 'C..."


### Mortality imbalance

In [177]:
print(f"Mortality at 30 days: {df['mortalidad_30_dias'].sum()/df.shape[0]*100:.2f}% of total records")  # target prevalence

Mortality at 30 days: 18.36% of total records


### Target and non-target columns

In [178]:
TARGET = "mortalidad_30_dias"

NONTARGETCOLS = ["record_id", "episode_id", "fecha_ingreso", "fecha_alta", "fecha_mortalidad", "mortalidad","dias_hemocultivo_mortalidad",
             "microorganismos_dict", "fechas_dict", "especimen_dict", "duracion_UCI","uci_por_el_episodio", "antimicrobiano_list", "cmi_list",
             "interpretacion_list", "antimicrobiano_family_list"]

df_ = df.drop(columns=NONTARGETCOLS) 

### Missing values heatmap

In [179]:
NA_THRESHOLD = 5

In [180]:
na_tbl = df_.isna().sum().sort_values(ascending=False)
missing_vars = []

for col in na_tbl[na_tbl > 0].index:
    pct = na_tbl[col] / len(df_) * 100
    if pct > NA_THRESHOLD:
        missing_vars.append(col)
        print(f"{col}: {na_tbl[col]} missing values --> {pct:.2f}% NaN")

puntaje_child_pugh: 3905 missing values --> 98.76% NaN
hepatopatia_moderada_o_grave: 3905 missing values --> 98.76% NaN
hepatopatia_ligera: 3905 missing values --> 98.76% NaN
somnolencia_estupor_coma: 3859 missing values --> 97.60% NaN
frecuencia_respiratoria: 3845 missing values --> 97.24% NaN
taquipnea: 3845 missing values --> 97.24% NaN
hipotermia_hipertermia: 3592 missing values --> 90.84% NaN
taquicardia: 3384 missing values --> 85.58% NaN
hipoxemia: 3381 missing values --> 85.51% NaN
barthel_inf_90: 3042 missing values --> 76.93% NaN
foco_controlable: 2365 missing values --> 59.81% NaN
disuria: 1489 missing values --> 37.66% NaN
tos: 1479 missing values --> 37.41% NaN
fiebre: 1479 missing values --> 37.41% NaN
dificultad_respirar: 1479 missing values --> 37.41% NaN
tenesmo_ano_recto: 1479 missing values --> 37.41% NaN
dolor_fosa_renal: 1479 missing values --> 37.41% NaN
nauseas: 1479 missing values --> 37.41% NaN
tenesmo_vejiga: 1479 missing values --> 37.41% NaN
vomitos: 1479 mi

In [181]:
# drop variables with too many missing values
df_ = df_.drop(columns=missing_vars)
print(f"Dropped {len(missing_vars)} variables with more than {NA_THRESHOLD}% missing values.")

Dropped 29 variables with more than 5% missing values.


In [182]:
# df dimensionality after dropping columns
print(f"Dataframe shape after dropping columns: {df_.shape}")

Dataframe shape after dropping columns: (3954, 102)


## Data Preprocessing

In [183]:
# Prepare data
X = df_.drop(columns=[TARGET])
y = df_[TARGET]

### Split data into train and tests sets
- Train / Validation: **70%** (80% train, 20% validation)
- Test: **30%**

In [184]:
# Split train (train+validation) and test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


## Model Finetuning

Selected models for finetuning (based on performance):
- Random Forest
- XGBoost
- LightGBM

### Random Forest

In [185]:
# Option A: single validation split (fast)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)


In [ ]:
# hyperparameter tuning 
param_dist_rf = {
    "n_estimators": [500, 800, 1200],
    "max_depth": [None, 4, 6, 8],
    "min_samples_leaf": [5, 10, 20, 40],
    "min_samples_split": [10, 20, 50],
    "max_features": ["sqrt", 0.3, 0.5],
}

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

search_rf = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=50,          
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/home/ppascual/micromamba/envs/bacthecom/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ppascual/micromamba/envs/bacthecom/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ppascual/micromamba/envs/bacthecom/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30

RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    n_jobs=-1,
                                                    random_state=42),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'max_depth': [None, 4, 6, 8],
                                        'max_features': ['sqrt', 0.3, 0.5],
                                        'min_samples_leaf': [5, 10, 20, 40],
                                        'min_samples_split': [10, 20, 50],
                                        'n_estimators': [500, 800, 1200]},
                   random_state=42, scoring='average_precision', verbose=1)

In [187]:
# Best hyperparameters
best_rf = search_rf.best_estimator_

print("Best CV PR-AUC:", search_rf.best_score_)
print("Best parameters:", search_rf.best_params_)
print("-------------")
results = pd.DataFrame(search_rf.cv_results_)
print(results.sort_values("mean_test_score", ascending=False).head(5)[['mean_test_score', 'std_test_score']])

Best CV PR-AUC: 0.4676043744669845
Best parameters: {'n_estimators': 800, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': None}
-------------
    mean_test_score  std_test_score
43         0.467604        0.053579
17         0.465554        0.050930
35         0.454933        0.055793
14         0.454811        0.055417
1          0.454635        0.053021


In [188]:
# fit best model hyperparameters
best_rf.fit(X_train, y_train)

# calibrate probabilities
calibrated_rf = CalibratedClassifierCV(best_rf, method='isotonic', cv=5)
calibrated_rf.fit(X_train, y_train)

y_prob = calibrated_rf.predict_proba(X_test)[:,1]

precision, recall, _ = precision_recall_curve(y_test, y_prob)

print(f"RF finetuned ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(f"RF finetuned PR-AUC: {average_precision_score(y_test, y_prob):.3f}")


RF finetuned ROC-AUC: 0.733
RF finetuned PR-AUC: 0.439


### LightGBM

In [189]:
# Option A: single validation split (fast)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)


In [190]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",  # used for pruning
        "boosting_type": "gbdt",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "verbosity": -1,
        "seed": 42,
    }

    # Training
    
    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_val, label=y_val)

    pruning_callback = optuna.integration.LightGBMPruningCallback(
        trial, metric="auc"
    )
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dvalid],
        callbacks=[
            pruning_callback,
            lgb.early_stopping(50, verbose=False),
        ]
    )

    y_pred_cv = model.predict(X_val, num_iteration=model.best_iteration)

    roc = roc_auc_score(y_val, y_pred_cv)
    pr  = average_precision_score(y_val, y_pred_cv)

    # Store metrics
    trial.set_user_attr("roc_auc", roc)
    trial.set_user_attr("pr_auc", pr)

    #print(f"Cross-Validation Trial {trial.number}: ROC={roc:.4f}, PR={pr:.4f}")

    return roc


In [191]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study_lgb = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)

study_lgb.optimize(objective, n_trials=100)


In [192]:
# fit best hyperparameter conf
best_params = study_lgb.best_trial.params
best_params.update({
    "objective": "binary",
    "metric": "auc",
    "verbosity": -1,
})

final_model = lgb.LGBMClassifier(
    **best_params,
    n_estimators=1000
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)


LGBMClassifier(colsample_bytree=0.6067537603871245,
               lambda_l1=0.00014612474911830902,
               lambda_l2=7.419608203150586e-08,
               learning_rate=0.02239633043895845, max_depth=8, metric='auc',
               min_child_samples=10, n_estimators=1000, num_leaves=240,
               objective='binary', subsample=0.7637683464430928, verbosity=-1)

### Predict optimized model on test set

In [193]:
y_pred = final_model.predict_proba(X_test)[:, 1]

roc_test = roc_auc_score(y_test, y_pred)
pr_test  = average_precision_score(y_test, y_pred)

summary = f"""
LIGHTGBM
Best trial (validation):
  ROC-AUC: {study_lgb.best_trial.user_attrs['roc_auc']:.4f}
  PR-AUC:  {study_lgb.best_trial.user_attrs['pr_auc']:.4f}

Test set:
  ROC-AUC: {roc_test:.4f}
  PR-AUC:  {pr_test:.4f}
"""

print(summary)



LIGHTGBM
Best trial (validation):
  ROC-AUC: 0.7738
  PR-AUC:  0.4097

Test set:
  ROC-AUC: 0.7380
  PR-AUC:  0.4525



### XGBoost - optuna

In [194]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

In [ ]:
best_pr = 0
best_roc = 0

def objective(trial):
    global best_pr, best_roc
    # Suggest hyperparameters
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",  # used for pruning
        "tree_method": "hist",  # fast
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
    }


    # Training
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_val, label=y_val)

    pruning_callback = optuna.integration.XGBoostPruningCallback(
        trial, "validation_0-auc"
    )

    evals_result = {}
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dvalid, "validation_0")],
        early_stopping_rounds=50,
        verbose_eval=False,
        callbacks=[pruning_callback],
        evals_result=evals_result
    )

    # Predictions
    y_pred_cv = model.predict(dvalid, iteration_range=(0, model.best_iteration))

    # Metrics
    roc = roc_auc_score(y_val, y_pred_cv)
    pr = average_precision_score(y_val, y_pred_cv)

    # Store metrics
    trial.set_user_attr("roc_auc", roc)
    trial.set_user_attr("pr_auc", pr)

    # Only print if new PR-AUC is higher than previous best
    if pr > best_pr:
        best_roc = roc
        best_pr  = pr
        print(f"New best trial {trial.number}: ROC={roc:.4f}, PR={pr:.4f}")

    # Return ROC-AUC for Optuna to maximize (pruning callback watches this)
    return roc


### Run optimization

In [196]:
# run optuna
study_xgb = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study_xgb.optimize(objective, n_trials=100)

In [197]:
# fit best hypermparametrization

best_params = study_xgb.best_trial.params
best_params.update({"objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist"})

final_model = xgb.XGBClassifier(
    **best_params,
    n_estimators=1000  # let early stopping during .fit() handle iterations
)
final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6236850165098831, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.016376844218916654,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=0.6152658633896286, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

### Predict optimized model on test set

In [198]:
y_pred = final_model.predict_proba(X_test)[:, 1]

roc_test = roc_auc_score(y_test, y_pred)
pr_test  = average_precision_score(y_test, y_pred)

summary = f"""
XGBOOST
Best trial (validation):
  ROC-AUC: {study_xgb.best_trial.user_attrs['roc_auc']:.4f}
  PR-AUC:  {study_xgb.best_trial.user_attrs['pr_auc']:.4f}

Test set:
  ROC-AUC: {roc_test:.4f}
  PR-AUC:  {pr_test:.4f}
"""

print(summary)



XGBOOST
Best trial (validation):
  ROC-AUC: 0.7761
  PR-AUC:  0.4647

Test set:
  ROC-AUC: 0.7322
  PR-AUC:  0.4419



### CatBoost

In [ ]:
# imbalance ratio 
neg = (y == 0).sum()
pos = (y == 1).sum()

imbalance_ratio = neg / pos
print(imbalance_ratio)


4.446280991735537


In [216]:
best_pr = 0
best_roc = 0

def objective(trial):
    global best_roc, best_pr  

    params_ctb = {
        "iterations": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 1.0),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "class_weights": [1, imbalance_ratio],  # imbalance handling
        "random_seed": 42,
        "verbose": False,
    }

    model = CatBoostClassifier(**params_ctb)

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )

    y_pred_cv = model.predict_proba(X_val)[:,1]
    roc = roc_auc_score(y_val, y_pred_cv)
    pr  = average_precision_score(y_val, y_pred_cv)

    trial.set_user_attr("roc_auc", roc)
    trial.set_user_attr("pr_auc", pr)

    # Only print if new PR-AUC is higher than previous best
    if pr > best_pr:
        best_roc = roc
        best_pr  = pr
        print(f"New best trial {trial.number}: ROC={roc:.4f}, PR={pr:.4f}")

    return roc


In [217]:
# run optuna
study_ctb= optuna.create_study(direction="maximize", pruner=None)
# study_xgb = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study_ctb.optimize(objective, n_trials=100)

New best trial 0: ROC=0.7194, PR=0.4213
New best trial 1: ROC=0.7457, PR=0.4706
New best trial 2: ROC=0.7680, PR=0.4785
New best trial 3: ROC=0.7710, PR=0.4864
New best trial 31: ROC=0.7660, PR=0.4869


In [223]:
# fit best hypermparametrization
best_params_ctb = study_ctb.best_trial.params

# Map tuned parameters to CatBoost
catboost_params = {
    "learning_rate": best_params_ctb["learning_rate"],
    "depth": best_params_ctb["depth"],
    "l2_leaf_reg": best_params_ctb["l2_leaf_reg"],
    "bagging_temperature": best_params_ctb["bagging_temperature"],
    "random_strength": best_params_ctb["random_strength"],
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "class_weights": [1, imbalance_ratio],
    "random_seed": 42,
    "verbose": False,
    "iterations": 1000,  # max iterations, early stopping will stop sooner
}

final_model = CatBoostClassifier(**catboost_params)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

In [226]:
best_params_ctb

{'learning_rate': 0.1852480471239771,
 'depth': 6,
 'l2_leaf_reg': 0.2810494195340575,
 'bagging_temperature': 0.6921170650307698,
 'random_strength': 0.22538674139586373}

In [225]:
y_pred = final_model.predict_proba(X_test)[:, 1]

roc_test = roc_auc_score(y_test, y_pred)
pr_test  = average_precision_score(y_test, y_pred)

summary = f"""
CATBOOST
Best trial (validation):
  ROC-AUC: {study_ctb.best_trial.user_attrs['roc_auc']:.4f}
  PR-AUC:  {study_ctb.best_trial.user_attrs['pr_auc']:.4f}

Test set:
  ROC-AUC: {roc_test:.4f}
  PR-AUC:  {pr_test:.4f}
"""

print(summary)



CATBOOST
Best trial (validation):
  ROC-AUC: 0.7774
  PR-AUC:  0.4820

Test set:
  ROC-AUC: 0.6946
  PR-AUC:  0.3983

